# Horarios Pico
**Proyecto:** Reserva Inteligente de Restaurantes — Etapa 3  
**Análisis:** Distribución de demanda por hora, día de semana y tipo de día  
**Fuente:** Data Warehouse Hive (`restaurant_dw`)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("horarios_pico_notebook")
    .master("spark://spark-master:7077")
    .config("spark.sql.warehouse.dir", "/opt/hive/data/warehouse")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.sql.catalogImplementation", "hive")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sql("USE restaurant_dw")
print("Spark version:", spark.version)

In [ ]:
# fact_pedido: select explícito para evitar colisión con columnas de partición anio/mes
fact_pedido = spark.table("fact_pedido").select(
    "id_tiempo", "id_restaurante", "id_usuario",
    "id_plato", "id_tipo_pedido", "id_estado_pedido",
    "id_pedido_origen", "cantidad", "subtotal",
    "latitud_entrega", "longitud_entrega",
)

fact_reservacion = spark.table("fact_reservacion").select(
    "id_tiempo", "id_restaurante", "id_usuario",
    "cant_personas", "duracion_minutos", "tasa_ocupacion", "estado",
)

# Renombrar id y nombre en dims para evitar ambigüedad
dim_tiempo      = spark.table("dim_tiempo").withColumnRenamed("id", "id_tiempo")
dim_restaurante = spark.table("dim_restaurante").withColumnRenamed("id", "id_restaurante").withColumnRenamed("nombre", "nombre_restaurante")

print(f"fact_pedido: {fact_pedido.count()} filas")
print(f"fact_reservacion: {fact_reservacion.count()} filas")

## 1. Demanda por hora del día

In [ ]:
df_hora = (
    fact_pedido
    .join(dim_tiempo,      "id_tiempo")
    .join(dim_restaurante, "id_restaurante")
    .groupBy(
        F.col("hora"),
        F.col("es_hora_pico"),
        F.col("es_fin_semana"),
        F.col("nombre_restaurante").alias("restaurante"),
    )
    .agg(
        F.countDistinct("id_pedido_origen").alias("total_pedidos"),
        F.countDistinct("id_usuario").alias("clientes_unicos"),
        F.round(F.sum("subtotal"), 2).alias("ingresos"),
    )
)

w_rest = Window.partitionBy("restaurante")
df_hora = df_hora.withColumn(
    "pct_pedidos_del_total",
    F.round(F.col("total_pedidos") * 100.0 / F.sum("total_pedidos").over(w_rest), 2)
).orderBy("restaurante", "hora")

df_hora.show(24, truncate=False)

## 2. Pico por día de semana × hora

In [ ]:
df_dia_hora = (
    fact_pedido
    .join(dim_tiempo,      "id_tiempo")
    .join(dim_restaurante, "id_restaurante")
    .groupBy(
        F.col("dia_semana"),
        F.col("nombre_dia"),
        F.col("hora"),
        F.col("es_fin_semana"),
        F.col("nombre_restaurante").alias("restaurante"),
    )
    .agg(
        F.countDistinct("id_pedido_origen").alias("total_pedidos"),
        F.round(F.sum("subtotal"), 2).alias("ingresos"),
    )
)

w_rank = Window.partitionBy("restaurante", "dia_semana").orderBy(F.desc("total_pedidos"))
df_dia_hora = (
    df_dia_hora
    .withColumn("rank_hora_en_dia", F.rank().over(w_rank))
    .orderBy("restaurante", "dia_semana", "hora")
)

df_dia_hora.filter(F.col("rank_hora_en_dia") <= 3).show(40, truncate=False)

## 3. Fin de semana vs días de semana

In [ ]:
df_tipo_dia = (
    fact_pedido
    .join(dim_tiempo,      "id_tiempo")
    .join(dim_restaurante, "id_restaurante")
    .withColumn(
        "tipo_dia",
        F.when(F.col("es_fin_semana"), F.lit("Fin de semana"))
         .otherwise(F.lit("Día de semana"))
    )
    .groupBy(
        F.col("tipo_dia"),
        F.col("nombre_restaurante").alias("restaurante"),
        F.col("hora"),
    )
    .agg(
        F.countDistinct("id_pedido_origen").alias("total_pedidos"),
        F.round(F.avg("subtotal"), 2).alias("ticket_promedio"),
        F.round(F.sum("subtotal"), 2).alias("ingresos_totales"),
    )
    .orderBy("restaurante", "tipo_dia", "hora")
)

df_tipo_dia.show(30, truncate=False)

## 4. Ocupación de mesas por hora

In [ ]:
df_ocupacion = (
    fact_reservacion
    .join(dim_tiempo,      "id_tiempo")
    .join(dim_restaurante, "id_restaurante")
    .filter(F.col("estado") == "reservada")
    .groupBy(
        F.col("hora"),
        F.col("es_fin_semana"),
        F.col("nombre_restaurante").alias("restaurante"),
    )
    .agg(
        F.count("*").alias("total_reservaciones"),
        F.round(F.avg("cant_personas"), 2).alias("personas_promedio"),
        F.round(F.avg("tasa_ocupacion"), 2).alias("ocupacion_promedio_pct"),
        F.round(F.avg("duracion_minutos"), 2).alias("duracion_promedio_min"),
    )
    .orderBy("restaurante", "hora")
)

df_ocupacion.show(24, truncate=False)

## 5. Guardar resultados

In [ ]:
import shutil, os

WAREHOUSE = "/opt/hive/data/warehouse/restaurant_dw.db"

def save_table(df, nombre):
    tabla = f"restaurant_dw.{nombre}"
    spark.sql(f"DROP TABLE IF EXISTS {tabla}")
    ruta = f"{WAREHOUSE}/{nombre}"
    if os.path.exists(ruta):
        shutil.rmtree(ruta, ignore_errors=True)
    df.write.mode("overwrite").saveAsTable(tabla)
    print(f"✅ {tabla}: {df.count()} filas guardadas.")

save_table(df_hora,     "resultado_demanda_por_hora")
save_table(df_dia_hora, "resultado_pico_dia_semana")
save_table(df_tipo_dia, "resultado_fin_semana_vs_semana")
save_table(df_ocupacion,"resultado_ocupacion_mesas_horaria")

spark.stop()